In [ ]:
import os
import json
from config import GAMES, DEFAULT_MAX_STEPS, DIFFICULTY_MAX_STEPS_MAP, NUM_EPISODES, LLM_BACKEND, LLM_MODEL, LLM_TEMPERATURE
from src.langmemm import build_graph
from src.utils import init_game_env, get_llm
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
import time
from src.schemas import SemanticFact, TopographicRelation, Step
from langmem import create_search_memory_tool, create_memory_store_manager


In [2]:
def run_single_game(game_name: str, game_path: str, difficulty:str, llm, max_steps: int, num_episodes: int = 1):
    env = init_game_env(game_path, max_steps=max_steps)
    episodic_store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small", 
    }
)   
    semantic_store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small",
    }
)   
    semantic_store_manager = create_memory_store_manager(
    "google_genai:gemini-2.0-flash",
    schemas=[SemanticFact, TopographicRelation],
    store=semantic_store,
    namespace=("memories", "semantic"),
    instructions=(
    "You are a memory manager for an agent that plays a text-based cooking game (TextWorld).\n"
"You receive observations from each game step, describing the environment, object properties, or spatial layout.\n"
"\n"
"Your task is to extract persistent knowledge from these observations.\n"
"\n"
"Extract:\n"
"- If the observation is a negative, repetitive, or uninformative message (e.g., 'You can't go that way!', 'Nothing happens.'), skip it or return a neutral placeholder.\n"
"- Use the 'SemanticFact' schema to extract general knowledge:\n"
"  - Object properties (e.g., 'the oven is closed')\n"
"  - Affordances (e.g., 'the knife can cut vegetables')\n"
"  - Item states (e.g., 'the cilantro is sliced')\n"
"  - Functional relationships (e.g., 'cookbook contains recipes')\n"
"- Use the 'TopographicRelation' schema to extract spatial and structural relationships:\n"
"  - Room connections (e.g., 'the kitchen is north of the pantry')\n"
"  - Containment (e.g., 'cilantro is in the fridge')\n"
"  - Object locations (e.g., 'knife is on the table')\n"
"\n"
"Extract facts that are likely to remain stable across time or episodes.\n"
"Ignore observations that only reflect temporary effects or single-step feedback.\n"
"Do not invent facts; only extract what is grounded in the observation."
)
)
    episodic_store_manager = create_memory_store_manager(
    "google_genai:gemini-2.0-flash",
    query_model="google_genai:gemini-2.0-flash",
    schemas=[Step],
    store=episodic_store,
    namespace=("memories", "episodic"),
    instructions=(
    "You are a memory manager for agent which plays a text-based cooking game (TextWorld). You receive the full context of a single gameplay step, including the observation before an action, the action itself, and the resulting observation.\n"
    "\n"
    "Extract episodic memory entries from agent interactions.\n"
    "- Use the 'Episode' schema to record each step.\n"
    "- Include:\n"
    "  - The observation before the action\n"
    "  - The exact action taken\n"
    "  - The reasoning behind the action (if available)\n"
    "  - The resulting observation or consequence\n"
    "  - Any score changes, inventory updates, or key environment feedback\n"
    "- Pay special attention to feedback indicating success (e.g., gaining points) or failure (e.g., losing resources, invalid moves).\n"
    "- Ignore generic or repeated messages unless they reflect meaningful change.\n"
    "- Store each step as a standalone entry in chronological order.\n"
    "- Do not modify or invent details not grounded in the input trace."
)
)

    config = RunnableConfig(recursion_limit=7*max_steps, configurable={"episodic_store_manager": episodic_store_manager,
                                                                        "semantic_store_manager": semantic_store_manager,})
    all_results = []
    game_log_dir = os.path.join("logs", "langmem", game_name)
    os.makedirs(game_log_dir, exist_ok=True)

    print(f"=============== Starting game: {game_name} [{difficulty}, {max_steps}] with {num_episodes} episodes ===============")

    for episode in range(num_episodes):
        graph = build_graph(llm=llm, env=env)
        print(f"\t=============== Running episode {episode + 1}/{num_episodes} ")
        obs, infos = env.reset()
        state = {

            "messages": [],
            "step": 1,
            "trace": [],
            "current_obs": obs,
            "score_count": 0,
            "done": False,
            "infos": infos,
            "episode_num": episode + 1,
        }

        state.setdefault("semantic_memory", "(no semantic memory retrieved)")
        state.setdefault("episodic_memory", "(no episodic memory retrieved)")
        state.setdefault("map_knowledge", "(no map knowledge retrieved)")

        start_time = time.time()
        output = graph.invoke(state, config=config)
        duration = time.time() - start_time

        trace = output.get("trace", [])
        total_tokens = sum(t.get("tokens_total", 0) for t in trace)
        total_prompt_tokens = sum(t.get("tokens_prompt", 0) for t in trace)
        total_completion_tokens = sum(t.get("tokens_completion", 0) for t in trace)

        
        steps_taken = output.get("step", 0)
        
        episode_result = {
            "episode": episode,
            "game": game_name,
            "difficulty": difficulty,
            "model_temperature": LLM_TEMPERATURE,
            "max_steps": max_steps,
            "agent": "langmem",
            "duration_seconds": round(duration, 2),
            "moves":  output.get("infos", 0).get("moves", 0),
            "max_score": output.get("infos", 0).get("max_score", 0),
            "won": output.get("infos", 0).get("won", 0),
            "lost": output.get("infos", 0).get("lost", 0),
            "score": output.get("score_count", 0),
            "steps_taken": steps_taken-1,
            "trace": output.get("trace", []),
            "total_tokens": total_tokens,
            "prompt_tokens": total_prompt_tokens,
            "completion_tokens": total_completion_tokens,
            "avg_tokens_per_step": round(total_tokens / steps_taken, 2) if steps_taken else 0,
            "trace": trace
        }

        # Save per-episode file
        episode_path = os.path.join(game_log_dir, f"episode_{episode}.json")
        with open(episode_path, "w") as f:
            json.dump(episode_result, f, indent=2)

        all_results.append(episode_result)
        print(f"\t=============== Finished episode {episode + 1}/{num_episodes}")
        print(f"""\tWon: {episode_result['won']}\n\tLost: {episode_result['lost']}\n\tScore: {episode_result['score']}\n\tMax Score: {episode_result["max_score"]}\n\tSteps Taken: {episode_result['steps_taken']}\n\tDuration: {episode_result['duration_seconds']} seconds""")

    print(f"All episodes completed for game: {game_name} [{difficulty}, {max_steps}].")
    print(f"Num won: {sum(1 for r in all_results if r['won'])}\nNum lost: {sum(1 for r in all_results if r['lost'])}")
    return all_results

def run_all_games():
    summary = []
    llm = get_llm(backend=LLM_BACKEND, model=LLM_MODEL, temperature=LLM_TEMPERATURE)

    for game in GAMES:
        name = game["name"]
        path = game["path"]
        difficulty = game["difficulty"]
        max_steps = DIFFICULTY_MAX_STEPS_MAP.get(difficulty, DEFAULT_MAX_STEPS)

        episodes = run_single_game(game_name=name, game_path=path, llm=llm, difficulty=difficulty, max_steps=max_steps, num_episodes=NUM_EPISODES)
        for ep_result in episodes:
            summary.append({"game": name, **ep_result})

    # Save summary
    os.makedirs("logs/langmem", exist_ok=True)
    with open("logs/langmem/summary.json", "a") as f:
        json.dump(summary, f, indent=2)

    print("Finished all games. Summary saved to logs/langmem/summary.json")


In [3]:
if __name__ == "__main__":
    run_all_games()

=============== Starting game: tw-cooking-recipe3+cook+cut+go9-XQ2ZC7bEH2ZBtMyW [very_hard, 45] with 4 episodes ===============
	=============== Running episode 1/4 
	=============== Finished episode 1/4
	Won: True
	Lost: False
	Score: 19
	Max Score: 4
	Steps Taken: 12
	Duration: 112.71 seconds
	=============== Running episode 2/4 
	Episode:2
Step:6
Observation:You open the copy of "Cooking: A Modern Approach (3rd Ed.)" and start reading:

Recipe #1
---------
Gather all following ingredients and follow the directions to prepare this tasty meal.

Ingredients:
  banana
  purple potato
  salt

Directions:
  grill the banana
  fry the purple potato
  prepare meal
Action:cook banana with BBQ
Result:You can't see any such thing.



Outcome:BLOCKED
Episode:1
Step:12
Observation:Can only prepare meal in the -= Kitchen =-.



Action:prepare meal
Result:Adding the meal to your inventory.


Your score has just gone up by one point.



Outcome:SUCCESS
Episode:0
Step:0
Observation:You are hungry! L

Could not apply patch: can't replace a non-existent object 'can_cook'


	=============== Finished episode 2/4
	Won: True
	Lost: False
	Score: 28
	Max Score: 4
	Steps Taken: 21
	Duration: 209.11 seconds
	=============== Running episode 3/4 
	Episode:2
Step:17
Observation:You open the BBQ.


Action:cook banana with BBQ
Result:You grilled the banana.


Your score has just gone up by one point.



Outcome:MILD_PROGRESS
Episode:3
Step:6
Observation:You can't see any such thing.

Action:take banana
Result:You already have that.

Outcome:BLOCKED
Episode:2
Step:16
Observation:-= Backyard =-
You've just walked into a backyard.

You see a patio chair. But there is nothing on it. You see a patio table. But there is nothing on it. You can see a closed BBQ here.

There is an open sliding patio door leading north. There is an open wooden door leading west. You don't like doors? Why not try going south, that entranceway is not blocked by one.
Action:open BBQ
Result:You open the BBQ.
Outcome:MILD_PROGRESS
Episode:3
Step:7
Observation:You already have that.

Action:cook ba

Could not apply patch: can't replace a non-existent object 'is_open'


	=============== Finished episode 1/4
	Won: True
	Lost: False
	Score: 104
	Max Score: 10
	Steps Taken: 24
	Duration: 172.52 seconds
	=============== Running episode 2/4 
	cookbook is_on
oven is closed
BBQ is open


...

	(no relevant memory found)
	Episode:1
Step:2
Observation:You take the cookbook from the counter.




Action:examine cookbook
Result:You open the copy of "Cooking: A Modern Approach (3rd Ed.)" and start reading:

Recipe #1
---------
Gather all following ingredients and follow the directions to prepare this tasty meal.

Ingredients:
  black pepper
  yellow onion

Directions:
  dice the yellow onion
  grill the yellow onion
  prepare meal




Outcome:MILD_PROGRESS
Episode:2
Step:1
Observation:\n\n                    ________  ________  __    __  ________        \n                   |        \\|        \\|  \\  |  \\|        \\       \n                    \\$$$$$$$$| $$$$$$$$| $$  | $$ \\$$$$$$$$       \n                      | $$   | $$__     \\$$\\/  $$   | $$          \n                      | $$   | $$  \\     >$$  $$    | $$          \n                      | $$   | $$$$$    /  $$$$\\    | $$          \n                      | $$   | $$_____ |  $$ \\$$\\   | $$          \n           

Could not apply patch: can't remove a non-existent object 'id'


	meal requires_action grill the diced yellow onion
oven is open


Could not apply patch: can't replace a non-existent object 'is_open'


	=============== Finished episode 3/4
	Won: False
	Lost: True
	Score: 15
	Max Score: 6
	Steps Taken: 11
	Duration: 85.36 seconds
	=============== Running episode 4/4 
	Episode:1
Step:2
Observation:You take the cookbook from the counter.




Action:examine cookbook
Result:You open the copy of "Cooking: A Modern Approach (3rd Ed.)" and start reading:

Recipe #1
---------
Gather all following ingredients and follow the directions to prepare this tasty meal.

Ingredients:
  black pepper
  yellow onion

Directions:
  dice the yellow onion
  grill the yellow onion
  prepare meal




Outcome:MILD_PROGRESS
Episode:3
Step:3
Observation:You open the copy of "Cooking: A Modern Approach (3rd Ed.)" and start reading:

Recipe #1
---------
Gather all following ingredients and follow the directions to prepare this tasty meal.

Ingredients:
  black pepper
  yellow onion

Directions:
  dice the yellow onion
  grill the yellow onion
  prepare meal
Action:take yellow onion
Result:You take the yellow onion